# binning

In [3]:
import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.model_selection  import train_test_split
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer


In [4]:
df=pd.read_csv("../../understanding-data/train.csv",usecols=["Age", "Fare", "Survived"])
df.sample(5)

,Survived,Age,Fare
494,0,21.0,8.0500
580,1,25.0,30.0000
729,0,25.0,7.9250
638,0,41.0,39.6875
885,0,39.0,29.1250


In [5]:
df.dropna(inplace=True)

In [6]:
x=df.iloc[:,1:]
y=df.iloc[:,0]

0      0
1      1
2      1
3      1
4      0
      ..
885    0
886    0
887    1
889    1
890    0
Name: Survived, Length: 714, dtype: int64

In [9]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,random_state=0)

In [10]:
clf = DecisionTreeClassifier()
clf.fit(x_train,y_train)
y_pred=clf.predict(x_test)
accuracy_score(y_pred,y_test)

0.6223776223776224

In [11]:
np.mean(cross_val_score(DecisionTreeClassifier(),x,y,cv=10,scoring="accuracy"))

np.float64(0.63589593114241)

In [20]:
kbin_age = KBinsDiscretizer(n_bins=10,encode="ordinal",strategy="quantile")
kbin_fare = KBinsDiscretizer(n_bins=10,encode="ordinal",strategy="quantile")

In [21]:
trf = ColumnTransformer([
    ("first",kbin_age,[0]),
    ("second",kbin_fare, [1])
])

In [22]:
x_train_trf=trf.fit_transform(x_train)
x_test_trf = trf.fit_transform(x_test)

/Users/keshavmacbook/Projects/AI-ML/AI-ML basic/.aiml-venv/lib/python3.13/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/Users/keshavmacbook/Projects/AI-ML/AI-ML basic/.aiml-venv/lib/python3.13/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/Users/keshavmacbook/Projects/AI-ML/AI-ML basic/.aiml-venv/lib/python3.13/site-p

In [23]:
trf.named_transformers_["second"].bin_edges_

array([array([  0.     ,   7.79664,   8.23998,  10.5    ,  13.     ,  16.     ,
               26.2575 ,  31.32   ,  65.96   ,  89.3    , 263.     ])          ],
      dtype=object)

In [24]:
output = pd.DataFrame({
    "age":x_train["Age"],
    "age_trf":x_train_trf[:,0],
    "fare":x_train["Fare"],
    "fare_trf":x_train_trf[:,1]
})

In [25]:
output["age_labels"]=pd.cut(x=x_train["Age"],bins= trf.named_transformers_["first"].bin_edges_[0].tolist())
output["fare_labels"]=pd.cut(x=x_train["Fare"],bins= trf.named_transformers_["second"].bin_edges_[0].tolist())

In [26]:
output

,age,age_trf,fare,fare_trf,age_labels,fare_labels
387,36.0,7.0,13.0000,4.0,"(33.0, 36.0]","(10.5, 13.0]"
685,25.0,4.0,41.5792,8.0,"(23.0, 26.0]","(31.32, 65.96]"
20,35.0,7.0,26.0000,6.0,"(33.0, 36.0]","(16.0, 26.258]"
331,45.5,8.0,28.5000,7.0,"(40.0, 50.0]","(26.258, 31.32]"
396,31.0,6.0,7.8542,1.0,"(29.0, 33.0]","(7.797, 8.24]"
...,...,...,...,...,...,...
883,28.0,5.0,10.5000,3.0,"(26.0, 29.0]","(8.24, 10.5]"
238,19.0,2.0,10.5000,3.0,"(13.2, 20.0]","(8.24, 10.5]"
789,46.0,8.0,79.2000,9.0,"(40.0, 50.0]","(65.96, 89.3]"
704,26.0,4.0,7.8542,1.0,"(23.0, 26.0]","(7.797, 8.24]"


In [27]:
clf = DecisionTreeClassifier()
clf.fit(x_train_trf,y_train)
y_pred=clf.predict(x_test_trf)
accuracy_score(y_pred,y_test)

0.6503496503496503

In [28]:
x_trf=trf.fit_transform(x)
np.mean(cross_val_score(DecisionTreeClassifier(),x_trf,y,cv=10,scoring="accuracy"))

/Users/keshavmacbook/Projects/AI-ML/AI-ML basic/.aiml-venv/lib/python3.13/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
/Users/keshavmacbook/Projects/AI-ML/AI-ML basic/.aiml-venv/lib/python3.13/site-packages/sklearn/preprocessing/_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


np.float64(0.682140062597809)

# binarization

In [29]:
df=pd.read_csv("../../understanding-data/train.csv",usecols=["Age", "Fare", "Survived","SibSp","Parch"])
df.sample(5)

,Survived,Age,SibSp,Parch,Fare
266,0,16.0,4,1,39.6875
362,0,45.0,0,1,14.4542
238,0,19.0,0,0,10.5000
151,1,22.0,1,0,66.6000
357,0,38.0,0,0,13.0000


In [30]:
df.dropna(inplace=True)

In [31]:
df["family"]= df["SibSp"]+df["Parch"]

In [32]:
df.sample(10)

,Survived,Age,SibSp,Parch,Fare,family
66,1,29.0,0,0,10.5000,0
35,0,42.0,1,0,52.0000,1
719,0,33.0,0,0,7.7750,0
879,1,56.0,0,1,83.1583,1
706,1,45.0,0,0,13.5000,0
129,0,45.0,0,0,6.9750,0
407,1,3.0,1,1,18.7500,2
620,0,27.0,1,0,14.4542,1
137,0,37.0,1,0,53.1000,1
590,0,35.0,0,0,7.1250,0


In [34]:
df.drop(columns=["SibSp","Parch"],inplace=True)

In [35]:
x=df.drop(columns=["Survived"])
y=df["Survived"]

In [36]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,random_state=0)

In [37]:
# without binarization
clf = DecisionTreeClassifier()
clf.fit(x_train,y_train)
y_pred=clf.predict(x_test)
accuracy_score(y_pred,y_test)

0.6083916083916084

In [ ]:
np.mean(cross_val_score(DecisionTreeClassifier(),x,y,cv=10,scoring="accuracy"))

np.float64(0.6471048513302035)

In [39]:
from sklearn.preprocessing import Binarizer

In [41]:
trf = ColumnTransformer([
    ("bin",Binarizer(copy=False),["family"])
],remainder="passthrough")

In [42]:
x_train_trf=trf.fit_transform(x_train)
x_test_trf=trf.fit_transform(x_test)

In [43]:
clf = DecisionTreeClassifier()
clf.fit(x_train_trf,y_train)
y_pred=clf.predict(x_test_trf)
accuracy_score(y_pred,y_test)

0.6293706293706294

In [45]:
x_trf=trf.fit_transform(x)
np.mean(cross_val_score(DecisionTreeClassifier(),x_trf,y,cv=10,scoring="accuracy"))

np.float64(0.6318466353677621)